# astrovodka Tutorial — From a TNG Subhalo to a Synthetic HI Data Cube

This tutorial walks you through the whole astrovodka workflow, **explaining why each step
is done** and where the pitfalls are. Everything in it runs out of the box on TNG-100
snapshot 99.

### What you will learn

| Section | Content |
|---|---|
| [1](#1-what-astrovodka-is) | What astrovodka is, and how it relates to MARTINI |
| [2](#2-environment-setup) | Environment setup |
| [3](#3-configuration) | How to choose a subhalo, two redshifts, two distances |
| [4](#4-loading-the-source-and-hi-models) | The four HI models, how to switch between them, how to read the results |
| [5](#5-building-the-mock-observation) | DataCube / beam / noise / spectral model, and verification |
| [6](#6-3d-data-products) | 3D grid products (UV field, DM/HI/H₂ density) |
| [7](#7-visualizing-the-data-cube) | Visualizing the data cube |
| [8](#8-a-trap-you-must-know-martini-prunes-the-source-in-place) | `Martini` prunes `source` in place — it comes last because its demonstration damages the source |
| [9](#9-troubleshooting) | Common errors and pitfalls |
| [10](#10-summary-and-next-steps) | Summary and next steps |

### Prerequisites

- **TNG-100 data**: this tutorial uses snapshot 099. The data path is passed in through the
  `TNG_BASE_PATH` environment variable, not hard-coded in the notebook:
  ```bash
  export TNG_BASE_PATH=/path/to/TNG/TNG-100/
  ```
- **Python environment**: `numpy`, `scipy`, `astropy`, `h5py` and `matplotlib` are required.
  See the [installation guide](../../docs/en/installation.md) for how to install them.

> **This tutorial's notebook stores no outputs.** The numbers in each section are printed at
> run time, so the results on your machine will match what is described here, and nothing
> depends on hard-coded values.


---

## 1. What astrovodka is

astrovodka turns the **gas particles** of a cosmological hydrodynamical simulation
(IllustrisTNG here) into **what a radio telescope would see** — a 21 cm data cube with
noise and a beam.

It is a fork of [MARTINI](https://github.com/kyleaoman/martini) (`astromartini` on PyPI)
that changes two things for high-redshift work.

### 1.1 The pipeline

```
                    ┌─────────────────────────────────────────────┐
   TNG snapshot ───▶│  TNGSource                                  │
   files            │   · reads gas/star/DM particles             │
   (groups_099,     │   · computes per-particle HI/H₂ masses  ◀── himodel
    snapdir_099)    │      with the selected HI model             │
                    │   · cosmological distances via Planck15     │
                    └───────────────────┬─────────────────────────┘
                                        │  mHI_g, positions, velocities,
                                        │  temperatures, smoothing lengths
                                        ▼
                    ┌─────────────────────────────────────────────┐
                    │  Martini   (orchestrates the observation)   │
                    │   insert_source_in_cube()  paint spectra    │
                    │   add_noise()              add noise        │
                    │   convolve_beam()          convolve beam    │
                    │   write_fits() / write_hdf5()               │
                    └───────────────────┬─────────────────────────┘
                                        ▼
                              DataCube ──▶ FITS cube
                    (pixel/channel grid + WCS sky coordinates)
```

Module responsibilities:

| Module | Responsibility |
|---|---|
| `astrovodka.himodel` | **HI/H₂ physics** (this package's own, four models) |
| `astrovodka.sources.TNGSource` | reads TNG data, assembles the source |
| `astrovodka.DataCube` | output grid and WCS |
| `astrovodka.Martini` | paint spectra / add noise / convolve beam / write files |
| `astrovodka.beams`, `noise`, `spectral_models`, `sph_kernels` | swappable physics components |

`himodel` and the observation layer are **one-way dependent**: the physics layer knows
nothing about data cubes, so it can be used on its own, outside the whole observation
pipeline (section 4.2 does exactly that).


### 1.2 The four HI models

This is astrovodka's main change relative to MARTINI. MARTINI uses the Leroy+2008
pressure calibration to split gas into HI and H₂; that is a calibration for nearby
galaxies, with **no redshift dependence**, so it is not appropriate at high redshift.

astrovodka implements the four methods of Diemer+2018 (arXiv:1806.02341),
i.e. {Gnedin & Draine 2014, Krumholz+ 2013} × {volumetric, projected}:

| `hi_model` | Where the surface density comes from | Source |
|---|---|---|
| `GD14_vol` | per-cell Jeans approximation, Σ = λ_J ρ | GD14 appendix C.3 |
| `GD14_map` | face-on 128² projection | ibid. + §2.3 |
| **`K13_vol`** (default) | per-cell Jeans approximation | K13 appendix C.4 |
| `K13_map` | face-on 128² projection | ibid. + §2.3 |

**Volumetric versions**: each gas cell uses its own density and internal energy to derive a
Jeans length, converts the volume density into a surface density, and then computes the
molecular fraction.

**Projected versions**: the galaxy is first projected into a face-on 2D map, and each cell
takes the molecular fraction of **the pixel it falls in**. They need stellar particles (to
compute the stellar surface density and velocity dispersion), and **gas outside the
projection box counts as zero HI mass** — the "galaxy" as defined by the model is that box.

### How to choose?

| Your goal | Which one |
|---|---|
| Default, least effort, results comparable with published work | `K13_vol` |
| Want to see systematic model differences (the comparison usually made in papers) | run all four, see section 4.2 |
| Want to avoid the Jeans approximation and measure the surface density directly | `*_map` |
| The source is very extended and you do not want to lose the outer gas | the volumetric versions; or set `map_clip_outside=True` for the map versions |

Measured deviations of Diemer+2018 from the official catalog (first 200 subhalos):

| Method | z=0 | z=0.5 | z=1.0 |
|---|---|---|---|
| GD14 vol | +4.0% | +3.0% | +1.9% |
| GD14 map | −5.0% | −10.6% | −12.2% |
| K13 vol | −8.0% | −0.3% | +1.6% |
| K13 map | −1.6% | −1.6% | −1.9% |


### 1.3 Changes relative to MARTINI

| Aspect | MARTINI | astrovodka |
|---|---|---|
| Distance | linear Hubble law `v = H₀·D` | `astropy.cosmology.Planck15`: D_L sets the flux, D_A sets the angular scale |
| Systemic velocity | `h·100·D` | radio convention `c·z/(1+z)`, consistent with the WCS spectral axis |
| HI model | pressure calibration (Leroy+2008) | four selectable Diemer+2018 models |
| UV field | — | per-galaxy adaptive FFT convolution, floored at the UVB |
| Data access | web API + JupyterLab | **local disk** (offline, fast) |
| Subhalo loading | the whole FOF group | optionally the subhalo alone (default, much faster) |
| Centring | the catalog's `SubhaloPos` | the **DM centre of mass** |
| H₂ | not provided | `source.m_H2_g`, `source.f_mol_g` |
| 3D products | — | `SubhaloDataProducts` |

> **There is no `import martini` anywhere.** MARTINI's observation machinery (data cube,
> beam, spectral model, SPH kernel, noise) is **lifted into this repository**, not an
> external dependency. `tests/test_selfcontained.py` enforces this with an AST parse.


---

## 2. Environment setup

In [ ]:
import os
import sys
import warnings
from pathlib import Path


def find_repo_root():
    """Locate the checkout, so the tutorial runs from any working directory."""
    override = os.environ.get('ASTROVODKA_ROOT')
    if override:
        return Path(override).resolve()
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / 'pyproject.toml').is_file() and (parent / 'astrovodka').is_dir():
            return parent
    return None


REPO_ROOT = find_repo_root()

# Use an installed astrovodka if there is one (pip install -e .); otherwise fall
# back to importing it straight from the checkout found above.
try:
    import astrovodka  # noqa: F401
except ModuleNotFoundError:
    if REPO_ROOT is None:
        raise ModuleNotFoundError(
            'astrovodka is not installed and no checkout was found. Run '
            '`pip install -e .` from the repository root, or set ASTROVODKA_ROOT '
            'to its path.'
        ) from None
    sys.path.insert(0, str(REPO_ROOT))

import h5py
import numpy as np
import astropy.units as U
from astropy.cosmology import Planck15
from astropy.io import fits
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Circle

# ===== astrovodka =====
from astrovodka import (Martini, DataCube, SubhaloDataProducts,
                        GroupDataProducts)
from astrovodka.group_products import ism_radius
from astrovodka.sources import TNGSource
from astrovodka.beams import GaussianBeam
from astrovodka.noise import GaussianNoise
from astrovodka.spectral_models import GaussianSpectrum
from astrovodka.sph_kernels import CubicSplineKernel
from astrovodka.cosmology_utils import (
    luminosity_distance, angular_diameter_distance, radio_velocity_from_redshift,
)
from astrovodka.himodel import (
    METHODS, DEFAULT_METHOD, HIModelConfig, calc_UMW_global_Faucher11,
    compute_hi_mass,
)

# low-level entry points to the physics layer, used in section 4.2 to compare the four models
from astrovodka.sources._illustris_tools import (
    loadHeader, loadSingle, loadSubset, getSnapOffsets,
)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150})

# The Chinese-language twin of this notebook picks a CJK-capable font at this
# point, walking through candidates such as 'Noto Sans CJK JP' and
# 'Source Han Sans SC' so that Chinese labels in the figures render as glyphs
# instead of tofu boxes. The English labels used here need no such handling.
plt.rcParams['axes.unicode_minus'] = False   # ASCII minus sign in tick labels

print('astrovodka', __import__('astrovodka').__version__)
print('numpy', np.__version__, '| astropy', U.__name__ and __import__('astropy').__version__)
print('Available HI models:', METHODS, '| default:', DEFAULT_METHOD)

---

## 3. Configuration

### 3.1 How to choose a subhalo (the first big pitfall)

**TNG's subhalo catalog is sorted by descending mass**, so the smaller the ID the larger the
mass:

- **ID 0 is the cluster core**, with about 38 million gas cells; a single `loadSubhalo` call
  peaks above 10 GB;
- when sampling a few objects for validation, **do not use `valid_ids[:N]`** — that selects
  cluster cores only;
- to thin the sample, use a fixed stride, or just use IDs in the 100~200 range.

This tutorial uses **subhalo 113**. It was chosen because the demonstration needs HI that is
**both abundant and actively star-forming**:

| | **113** (this tutorial) | 139 | 122 |
|---|---|---|---|
| Gas cells | 8169 | 7246 | 4448 |
| M_HI [M☉] | 4.3e9 | **4.5e9** | 3.3e9 |
| M_H₂ [M☉] | **1.8e9** | 3.0e8 | 1.9e8 |
| SFR [M☉/yr] | 0.98 | 0.17 | 0.19 |
| UV field peak / background | **27.6 / 0.0014 (≈19000×)** | 2.2 | 6.0 |
| HI difference, `adaptive_fft` vs `global` | **+13.9%** | +5.7% | +4.3% |

139 has 5% more HI in total, but it barely forms stars, and its UV field peaks at only 1550
times the background — a demonstration built on it would show almost nothing of the adaptive
FFT UV field. With 113 you can see a 14% difference.

> Changing the subhalo means changing only the single `subID` variable below. Small galaxies
> that run quickly (about 4000~9000 cells) are mostly in the 100~200 range.

In [ ]:
# ===================== Configuration =====================
# The TNG data path is read from an environment variable, not hard-coded in the notebook:
#     export TNG_BASE_PATH=/path/to/TNG/TNG-100/
basePath = os.environ.get('TNG_BASE_PATH', '')
if not basePath:
    raise RuntimeError(
        'Please tell the notebook where the TNG-100 data directory is:\n'
        '    export TNG_BASE_PATH=/path/to/TNG/TNG-100/\n'
        '(the directory should contain groups_099/ and snapdir_099/)'
    )
basePath = basePath.rstrip('/') + '/'

snapNum  = 99        # TNG-100 snapshot, z ≈ 0
subID    = 113       # see the previous section: plenty of HI and actively star-forming
Z_OBS    = 0.02      # the observed redshift at which to place the simulation (see 3.2)
HI_MODEL = 'K13_vol' # one of four: K13_vol / K13_map / GD14_vol / GD14_map

# Section 6.3: the FOF group. Subhalo 113 lives in group 0, the biggest cluster
# in the box -- not something to load in a notebook, so a smaller group stands in.
# See 6.3 for how to choose one.
GROUP_ID    = 59    # FOF group analysed in section 6.3
GROUP_NGRID = 192   # grid edge length for the group products

# Products go to examples/outputs/ (already in .gitignore); the path is resolved
# relative to the repository root.
OUTDIR = str((REPO_ROOT or Path.cwd()) / 'examples' / 'outputs') + '/'
os.makedirs(OUTDIR, exist_ok=True)

if not os.path.isdir(basePath + f'groups_{snapNum:03d}'):
    raise RuntimeError(
        f'{basePath}: groups_{snapNum:03d}/ not found, please check TNG_BASE_PATH.'
    )

cosmo = Planck15

# ---- the geometry implied by this configuration ----
DL = luminosity_distance(Z_OBS, cosmo)
DA = angular_diameter_distance(Z_OBS, cosmo)
arcsec_to_kpc = (DA.to(U.kpc) / U.rad).to(U.kpc / U.arcsec)

print(f'Observed redshift          z    = {Z_OBS}')
print(f'Luminosity distance        D_L  = {DL:.1f}   ← sets the flux')
print(f'Angular diameter distance  D_A  = {DA:.1f}   ← sets the angular scale')
print(f'Scale                      1"   = {arcsec_to_kpc.value:.3f} kpc')
print(f'Systemic velocity          vsys = {radio_velocity_from_redshift(Z_OBS):.0f}')
print(f'Output directory                = {OUTDIR}')

### 3.2 Snapshot redshift vs observed redshift

This is the easiest thing to mix up, and `TNGSource` keeps the two **separate**:

| | Where it comes from | What it is used for |
|---|---|---|
| **Snapshot redshift** `z_snapshot` | the snapshot header | **physics**: unit conversions, the HI model. You must use the simulation's own redshift, otherwise every mass is wrong |
| **Observed redshift** `observed_redshift` | the parameter you pass | **geometry**: D_L, D_A, vsys. It decides "how far away the galaxy is placed to be observed" |

If you do not pass `observed_redshift`, the two are the same — for snapshot 99 (z≈0) that
means D_A is almost zero, the galaxy shrinks to a point on the sky, and there is nothing in
the cube.

Passing `observed_redshift=0.02` means **observing a z≈0 galaxy's physics at a distance of
z=0.02**. This is a common practice in high-redshift work: the simulation itself does not
reach that far, but the telescope's angular resolution and sensitivity require the galaxy to
sit at some distance.

Below we look at the same galaxy placed at different distances.

**Note** that variables such as `HI_MODEL` and `subID` were defined in the previous cell, and
`source` does not exist yet; this only computes geometry, and does not touch the data.

In [ ]:
rows = []
v_snapshot = radio_velocity_from_redshift(2.2e-16)   # redshift of snapshot 99

for z in [0.0, 0.005, 0.02, 0.05, 0.1]:
    dl = luminosity_distance(z, cosmo)
    da = angular_diameter_distance(z, cosmo)
    # the same galaxy at different distances: its angular size on the sky scales as 1/D_A
    rows.append((z, dl.value, da.value, (da / DA).value,
                 radio_velocity_from_redshift(z)))

print(f'{"z_obs":>7} {"D_L [Mpc]":>11} {"D_A [Mpc]":>11} '
      f'{"angular scale rel. z=0.02":>18} {"vsys [km/s]":>12}')
print('-' * 64)
for z, dl, da, rel, v in rows:
    print(f'{z:>7.3f} {dl:>11.1f} {da:>11.1f} {rel:>18.3f} {v:>12.0f}')

print(f'\nThe galaxy itself has z ≈ 0; vsys in the radio convention c·z/(1+z) = {v_snapshot:.1f} km/s')
print('At z=0 D_A hits its floor of ~4.4 kpc, and astrovodka raises a UserWarning to tell you.')

### 3.3 Two distances, each with its own job

This is a major bug that was fixed in v0.2.0 (at the time D_A was computed but never used,
so every angular scale was too small by (1+z)²).

| Purpose | Which distance | Why |
|---|---|---|
| Sky position, RA/Dec, angular scale, angular width of the SPH kernel | **D_A** | angular scale = physical scale / D_A |
| Normalisation of the flux | **D_L** | S ∝ M / D_L² |
| Systemic velocity `vsys` | radio convention `c·z/(1+z)` | consistent with the WCS spectral axis `VRAD` |

The flux line is off by (1+z)⁴ — use the wrong distance and the cube's total flux is not what
this galaxy should have. You do not need to handle this by hand: `TNGSource` already wires
the three together. But when reading the code, know that `source.skycoords.distance` is
**D_A**, while `source.luminosity_distance` is **D_L**.

---

## 4. Loading the source and HI models

A single class, `TNGSource`, does four jobs: read the snapshot, compute distances, compute HI
masses, and hand everything over to `SPHSource`.

```python
source = TNGSource(
    basePath=..., snapNum=99, subID=113,
    hi_model='K13_vol',      # any one of the four models
    hi_config=None,          # HIModelConfig or dict, see 4.3
    uv_mode='adaptive_fft',  # volumetric versions only, see 4.4
    observed_redshift=0.02,  # observed redshift, see 3.2
    load_subhalo_only=True,  # True reads the subhalo only, False reads the whole FOF group
    ra=30*U.deg, dec=45*U.deg,
)
```

`hi_model` is validated **before anything is read from disk**, so a typo does not cost you a
multi-GB load.

> **When is `load_subhalo_only=False` needed?** When foreground/background gas towards the
> target galaxy would fall inside the cube's field of view. The default `True` reads only
> this subhalo's own particles, which is much faster, but for a satellite galaxy in a cluster
> it may miss gas belonging to other members of the same group.

In [ ]:
if HI_MODEL not in METHODS:
    raise ValueError(f'HI_MODEL must be one of {METHODS}, got {HI_MODEL!r}')

print(f'Loading TNG-100 snapshot {snapNum} subhalo {subID} ...')
source = TNGSource(
    basePath=basePath,
    snapNum=snapNum,
    subID=subID,
    hi_model=HI_MODEL,
    observed_redshift=Z_OBS,
    load_subhalo_only=True,
    ra=30.0 * U.deg,
    dec=45.0 * U.deg,
)

print(f'\n=== Source overview ===')
print(f'  Particle counts: gas {source.n_gas}, stars {source.n_star}, DM {source.n_dm}')
print(f'  Snapshot redshift {source.source_redshift:.6f}  →  observed redshift {Z_OBS}')
print(f'  D_L = {source.D_L:.1f}   D_A = {source.D_A:.1f}')
print(f'  vsys = {source.vsys.to(U.km/U.s):.0f}')
print(f'\n=== Mass budget ===')
print(f'  HI    {source.m_HI_total:.4e}')
print(f'  H2    {source.m_H2_g.sum():.4e}')
print(f'  Gas   {source.m_gas_total:.4e}')
print(f'  Stars {source.m_star_total:.4e}')
print(f'  DM    {source.m_dm_total:.4e}')
print(f'  Neutral gas as a fraction of the total gas {100*(source.m_HI_total + source.m_H2_g.sum())/source.m_gas_total:.1f}%')
print(f'\n  Model used: {source.hi_model}   ({source.hi_model_spec["reference"]})')
print(f'  Model configuration: {source.hi_config.summary()}')
print(f'  Is it a projected model: {source.is_projected}')

# ---- angular size and velocity width of the source: they decide how big the cube must be ----
xyz  = source.coordinates_g.get_xyz()
vxyz = source.coordinates_g.differentials['s'].get_d_xyz()

y_kpc = 1.1 * (xyz[1].max() - xyz[1].min()).to_value(U.kpc)
z_kpc = 1.1 * (xyz[2].max() - xyz[2].min()).to_value(U.kpc)
v_kms = 1.1 * max((vxyz[0].max() - vxyz[0].min()).to_value(U.km/U.s), 20.0)

y_extent_arcsec = y_kpc / arcsec_to_kpc.value
z_extent_arcsec = z_kpc / arcsec_to_kpc.value
print(f'\n=== Source extent (sets the cube field of view, 1.1× the particle range) ===')
print(f'  Sky Y (RA)   {y_extent_arcsec:6.1f}"  = {y_kpc:6.1f} kpc')
print(f'  Sky Z (Dec)  {z_extent_arcsec:6.1f}"  = {z_kpc:6.1f} kpc')
print(f'  Line-of-sight velocity spread {v_kms:6.0f} km/s')
print(f'\n  In other words: the field of view must be at least {max(y_extent_arcsec, z_extent_arcsec):.0f}" to hold the whole galaxy,')
print(f'  otherwise Martini will prune particles away (section 5.4 shows this happening).')

### 4.1 What the model actually computes

Besides `source.m_HI_total`, all the per-particle intermediate quantities are kept as well —
very useful for debugging and for physical interpretation:

| Attribute | Shape | Meaning |
|---|---|---|
| `source.mHI_g` | (n_gas,) | per-particle HI mass [M☉] |
| `source.m_H2_g` | (n_gas,) | per-particle H₂ mass [M☉] |
| `source.f_mol_g` | (n_gas,) | per-cell **molecular fraction** f_H₂/(f_HI+f_H₂) |
| `source.f_neutral_g` | (n_gas,) | per-cell neutral fraction (1 for star-forming cells) |
| `source.u_mw_g` | (n_gas,) | per-cell UV field [Draine units, 1 = Milky Way] |
| `source.hi_diagnostics` | dict | the model's own diagnostics (projection geometry, alignment, ...) |
| `source.hi_result` | `HiMassResult` | the container for all of the above, plus the `cells` table |

Plotting the molecular fraction and the UV field shows directly what the model is doing.

> Each point in the figures is one gas cell. The star-forming cells (f_neutral = 1, yellow in
> panel (c)) have high density and a strong UV field, but they also have the **highest
> molecular fraction** — because what sets f_mol is the surface-density term τ_c, so dense
> gas stays mostly molecular even under strong UV illumination.
>
> So panel (c) **looks** like "the stronger the UV, the more molecules", but that is an
> illusion created by density and UV being positively correlated in a real galaxy. The
> **anti**-correlation that the UV field has in the model (the χ term of K13) is only visible
> at **fixed density** — which is exactly how `test_more_uv_means_fewer_molecules` in
> `tests/test_hi_models.py` tests it.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

u   = np.asarray(source.u_mw_g, dtype='float64')
fm  = np.asarray(source.f_mol_g, dtype='float64')
fne = np.asarray(source.f_neutral_g, dtype='float64')
sfr = np.asarray(source.sfr_g, dtype='float64')
uvb = calc_UMW_global_Faucher11(source.source_redshift)  # the true UVB background

# (a) per-cell UV field
ax = axes[0]
ax.hist(u, bins=np.logspace(np.log10(max(u.min(), 1e-8)), np.log10(u.max()), 50),
        color='C3', alpha=.85)
ax.set_xscale('log')
ax.axvline(uvb, color='k', ls='--', lw=1.2, label=f'UVB background {uvb:.2e}')
ax.set_xlabel('U_MW  [Draine units]')
ax.set_ylabel('number of cells')
ax.set_title('(a) per-cell UV field')
ax.legend()

# (b) distribution of the molecular fraction
ax = axes[1]
ax.hist(fm, bins=50, color='C0', alpha=.85)
ax.set_yscale('log')
ax.set_xlabel(r'$f_\mathrm{mol} = f_{H_2}/(f_{HI}+f_{H_2})$')
ax.set_ylabel('number of cells')
ax.set_title('(b) molecular fraction distribution')

# (c) f_mol vs U_MW, coloured by the neutral fraction
ax = axes[2]
sc = ax.scatter(u, fm, c=fne, s=7, alpha=.55, cmap='viridis')
ax.set_xscale('log')
ax.set_xlabel('U_MW  [Draine units]')
ax.set_ylabel(r'$f_\mathrm{mol}$')
ax.set_title('(c) f_mol vs UV field')
plt.colorbar(sc, ax=ax, label=r'$f_\mathrm{neutral}$', shrink=.85)

for ax in axes:
    ax.grid(alpha=.2)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, f'sub{subID}_z{Z_OBS}_model_diagnostics.png'),
            bbox_inches='tight', facecolor='white')
plt.show()

# mass-weighted H2 fraction (not a simple average of f_mol)
h2_frac = source.m_H2_g.sum() / (source.m_HI_total + source.m_H2_g.sum())
print(f'UV field       {u.min():.4g} .. {u.max():.4g}')
print(f'  UVB background {uvb:.4g}   →   the peak is {u.max()/uvb:.0f} times the background')
print(f'  (everywhere in this galaxy exceeds the background, so the per-galaxy FFT field is really at work)')
print(f'f_mol          median {np.median(fm):.4f}, max {fm.max():.4f}')
print(f'H2 mass fraction {100*h2_frac:.1f}%')
print(f'Star-forming cells {int((sfr > 0).sum())} / {source.n_gas}')

### 4.2 Switching models: run all four

Every `TNGSource` construction re-reads the data from disk, so the cheaper way to compare the
four models is to **read the data once and call the physics layer four times**:

```python
from astrovodka.himodel import HIModelConfig, compute_hi_mass

cfg = HIModelConfig(h=..., z=...)          # h, z come from the snapshot
res = compute_hi_mass(gas, model='K13_map', cfg=cfg, stars=stars, sub=sub)

res.m_hi          # per cell, TNG code units (1e10 M☉/h), length = number of gas cells
res.m_hi_msun     # total, M☉
res.diagnostics   # projection geometry, alignment, in/out-of-box gas fractions, ...
```

The `model` argument of `compute_hi_mass` is case-insensitive; `'K13'`/`'GD14'` mean the
volumetric versions, and two-element forms such as `('K13','map')` are accepted too.

> **This is the payoff of the one-way dependency of `himodel`**: the physics layer does not
> need a data cube, so it can be called and tested on its own, outside the whole observation
> pipeline.

In the next cell, `gas`/`stars`/`sub` are the raw fields read straight from the snapshot with
the same readers that `TNGSource` uses internally — so the four computations share one copy of
the data.

In [ ]:
# ---- read the data once, shared by the four models ----
hdr = loadHeader(basePath, snapNum)
h_sim, z_sim = float(hdr['HubbleParam']), float(hdr['Redshift'])
cfg = HIModelConfig(h=h_sim, z=z_sim, boxsize_ckpc_per_h=float(hdr['BoxSize']))

subset = getSnapOffsets(basePath, snapNum, subID, 'Subhalo')

gas_fields  = ['Masses', 'Velocities', 'InternalEnergy', 'ElectronAbundance',
               'NeutralHydrogenAbundance', 'StarFormationRate', 'Density',
               'Coordinates', 'CenterOfMass', 'GFM_Metals', 'GFM_Metallicity']
gas_mdi     = [None]*9 + [0] + [None]          # GFM_Metals: only the H column
star_fields = ['Masses', 'Coordinates', 'Velocities', 'GFM_StellarFormationTime']
gas_raw  = loadSubset(basePath, snapNum, 'gas',  fields=gas_fields,
                      subset=subset, mdi=gas_mdi)
star_raw = loadSubset(basePath, snapNum, 'star', fields=star_fields, subset=subset)

# the projected models need the subhalo's own position, velocity and half-mass radii (raw ckpc/h)
row   = loadSingle(basePath, snapNum, subhaloID=subID)
half  = np.asarray(row['SubhaloHalfmassRadType'], dtype='float64')
sub   = dict(pos=np.asarray(row['SubhaloPos'], dtype='float64'),
             vel=np.asarray(row['SubhaloVel'], dtype='float64'),
             rhalf_gas=float(half[0]), rhalf_star=float(half[4]))

print(f'Read {gas_raw["count"]} gas cells and {star_raw["count"]} star particles')
print(f'R½(gas) = {half[0]:.1f} ckpc/h,  R½(star) = {half[4]:.1f} ckpc/h\n')

# ---- run each of the four models once ----
results = {}
for m in METHODS:
    kw = dict(stars=star_raw, sub=sub) if m.endswith('_map') else {}
    results[m] = compute_hi_mass(gas_raw, m, cfg, **kw)

# ---- comparison table ----
print(f'{"model":<10} {"M_HI [M☉]":>13} {"M_H2 [M☉]":>13} {"M_neutral [M☉]":>13} '
      f'{"f_mol":>7} {"effective cells":>9}')
print('-' * 72)
for m, r in results.items():
    n_eff = int(np.count_nonzero(r.m_neutral))
    fmol = r.m_h2_msun / r.m_neutral_msun if r.m_neutral_msun else 0.0
    print(f'{m:<10} {r.m_hi_msun:>13.4e} {r.m_h2_msun:>13.4e} '
          f'{r.m_neutral_msun:>13.4e} {fmol:>7.3f} {n_eff:>9d}')

hi_vol = results['K13_vol'].m_hi_msun
print(f'\nHI deviation relative to K13_vol:')
for m, r in results.items():
    print(f'  {m:<10} {100*(r.m_hi_msun/hi_vol - 1):+6.1f}%')

print('\nNote that the two volumetric models have **exactly the same total neutral mass** — the')
print('models differ only in how that neutral gas is divided between HI and H2, not in how\n'
      'much neutral hydrogen there is. The projected versions come out a little lower because\n'
      'gas outside the projection box is recorded as zero mass.')

### 4.3 Tuning the model: `HIModelConfig`

`HIModelConfig` is a frozen dataclass holding `h`, `z` and all the numerical switches.

```python
cfg = HIModelConfig(h=0.6774, z=0.0, ngrid=128)
```

When passing it to `TNGSource` you can use a dict (overriding only what you want to change),
or a complete object:

```python
TNGSource(..., hi_config={'ngrid': 64, 'k13_variant': 'hydro'})
```

> **`h` and `z` always follow the snapshot.** Any `h`/`z` you write in `hi_config` are
> overridden by the snapshot header values, because interpreting this data with another
> snapshot's units is meaningless, and a silent error of that kind is invisible anyway. The
> configuration actually in force is in `source.hi_config`.

The main switches:

| Field | Default | Meaning |
|---|---|---|
| `ngrid` | 128 | edge length of the 3D FFT UV field grid |
| `f_esc` | 0.1 | UV escape fraction (the calibrated value of Diemer+2018 §4.2) |
| `box_expand` | 1.1 | adaptive UV box = this factor × the gas coordinate span |
| `ngrid_map` | 128 | edge length of the projection grid |
| `vol_fine_grid` | 0.0 | the second, fine UV grid of the paper's App. A.2; **off by default** |
| `map_clip_outside` | `False` | `True` = clip gas outside the projection box to the boundary pixels (mass conserving) |
| `k13_variant` | `'2p'` | how n_CNM is obtained in the projected K13 model |
| `rho_dm_mode` | `'none'` | dark matter mid-plane surface density for the projected K13 model |

A few worth knowing about:

- **`map_clip_outside`**: with the default `False`, the HI of gas outside the projection box
  is recorded as zero. Setting it to `True` clips that gas to the boundary pixels: mass is
  conserved, but it piles up at the edges.
- **`vol_fine_grid`**: the paper says a second, fine grid is needed, otherwise the UV field at
  the galaxy centre is underestimated. But the paper does not say how the two grids are
  combined, and turning it on **breaks bit-for-bit agreement with the reference
  implementation**, so it is off by default — it is an experiment switch, not a bug fix.
- **`k13_variant`**: the official K13_map products use `'2p'`; `'hydro'` (the form used in the
  paper's main text) runs away to f_mol = 1 through positive feedback, and is measured to
  differ by three orders of magnitude. All three variants are kept for comparison.

In [ ]:
# ---- run with a different configuration to see how the parameters affect the result ----
print('K13_map with different n_CNM prescriptions:')
print(f'{"k13_variant":<14} {"M_HI [M☉]":>13} {"M_H2 [M☉]":>13}')
print('-' * 44)
for variant in ('2p', 'hydro', 'hydro_nostar'):
    c = cfg.with_(k13_variant=variant)
    r = compute_hi_mass(gas_raw, 'K13_map', c, stars=star_raw, sub=sub)
    print(f'{variant:<14} {r.m_hi_msun:>13.4e} {r.m_h2_msun:>13.4e}')
print("\n→ The official products correspond to '2p' (the default). 'hydro' is the form in")
print('  appendix C.4 of the paper, but its hydrostatic pressure floor drives the iteration\n'
      '  into a positive feedback loop towards f_mol=1, orders of magnitude away from the\n'
      '  official products.')

# ---- mass conservation audit: inside and outside the projection box ----
r_map = results['K13_map']
d = r_map.diagnostics
print(f'\nProjection geometry (K13_map):')
print(f'  projection box edge L = {d["L_map_kpc"]:.1f} kpc, pixel {d["pix_kpc"]*1e3:.0f} pc')
print(f'  cells inside the box {d["n_cells_kept"]} / {d["n_cells_total"]}')
print(f'  gas mass fraction inside the box f_gas_in_map = {d["f_gas_in_map"]:.4f}')
print(f'  alignment succeeded = {bool(d["align_success"])} (branch: {d["align_branch"]}, '
      f'using {d["n_gas_align"]} gas cells / {d["n_star_align"]} star particles)')
if 'closure_m_h2' in d:
    print(f'  per-cell re-sum vs image integral closure: H2 {d["closure_m_h2"]:.6f}, '
          f'HI {d["closure_m_hi"]:.6f}   (1.0 = fully self-consistent)')
print('\n→ alignment vector (face-on):', np.round(d['align_ang_mom_vector'], 4))

### 4.4 The UV field: `uv_mode` (only meaningful for the volumetric versions)

The volumetric versions have two ways of obtaining the UV field:

| `uv_mode` | Meaning |
|---|---|
| `'adaptive_fft'` (default) | FFT-convolve the star-formation-rate distribution to get a per-galaxy UV field, then floor it at the UVB |
| `'global'` | use only the uniform Faucher-Giguère+2011 ultraviolet background |

Both **floor at the background**, so `'global'` is the limit in which "this galaxy forms no
stars at all".

**The projected versions do not take this parameter** — their UV field comes from a 2D
convolution of the SFR surface-density map, which is part of the model definition. Passing
`uv_mode='global'` to a `*_map` model raises an error immediately, rather than being silently
ignored.

In [ ]:
if source.is_projected:
    print(f'{HI_MODEL} is a projected model, uv_mode does not apply — passing global raises:')
    try:
        TNGSource(basePath=basePath, snapNum=snapNum, subID=subID,
                  hi_model=HI_MODEL, uv_mode='global')
    except ValueError as e:
        print(f'  ValueError: {e}')
else:
    r_fft = compute_hi_mass(gas_raw, HI_MODEL, cfg, uv_mode='adaptive_fft')
    r_uvb = compute_hi_mass(gas_raw, HI_MODEL, cfg, uv_mode='global')
    print(f'{HI_MODEL} with the two UV prescriptions:')
    print(f'  {"uv_mode":<14} {"M_HI [M☉]":>13} {"M_H2 [M☉]":>13} {"U_MW peak":>11}')
    print('  ' + '-' * 56)
    for name, r in (('adaptive_fft', r_fft), ('global', r_uvb)):
        print(f'  {name:<14} {r.m_hi_msun:>13.4e} {r.m_h2_msun:>13.4e} '
              f'{r.u_mw.max():>11.4g}')
    print(f'\n  HI differs by {100*(r_fft.m_hi_msun/r_uvb.m_hi_msun - 1):+.1f}%, '
          f'H2 differs by {100*(r_fft.m_h2_msun/r_uvb.m_h2_msun - 1):+.1f}%')
    print('  → The more active the star formation, the larger this difference. This is the bug')
    print('    fixed in v0.3.0: the old interpolator shared one grid across all three axes, so\n'
          '    adaptive_fft degenerated to global in practice.')

---

## 5. Building the mock observation

By now `source` is ready. Next we define the **observation**: what grid the telescope sees,
what beam, what noise.

```
DataCube (grid + WCS)  +  Beam  +  Noise
        +  SpectralModel (line shape)  +  SPHKernel (particle deposition)
                              │
                              ▼
                          Martini
```

Every component is swappable: swap `GaussianBeam` for another telescope, swap
`CubicSplineKernel` for another deposition scheme.

### 5.1 `DataCube`: pixels, channels, spectral axis units

```python
DataCube(n_px_x, n_px_y, n_channels,
         px_size, channel_width, spectral_centre,
         ra, dec, spectral_unit=None)
```

Three quantities must be set from the source's extent, not guessed:

| Parameter | How to set it | This tutorial |
|---|---|---|
| `px_size` | sample at least 3 pixels per beam FWHM (Nyquist) | beam/3 |
| `n_px_x/y` | field of view ≥ the source's angular size (otherwise particles are pruned, see 5.4) | 1.1× the source extent |
| `channel_width` | the channel range must cover **the particle velocity range + the line half-width ±4σ** (see 5.4), then be split into a few tens of channels | see the computation in the next cell |

### ⚠️ `spectral_unit`: a silent bug fixed in v0.2.0

`spectral_unit=None` (the default) means **keep the unit of `channel_width`**, i.e.
`20*U.km/U.s` gives a velocity axis and `100*U.kHz` gives a frequency axis.

If you pass `'MHz'` explicitly while `channel_width` is still in velocity units, it is
converted to frequency as a **difference**. The old version used astropy's `doppler_radio`
equivalence here — that is a mapping from **absolute** velocity to **absolute** frequency, so
using it on a *width* returns `f_rest` itself rather than a small quantity, and the cube was
left with only 0.007% of the flux. **MHz and km/s now give exactly the same flux.**

This is verified in the last section of this tutorial — it runs only there because the small
verification cube **prunes `source` in place**, which is itself worth its own section
(see section 8).

In [ ]:
# ===== fix the geometry first: field of view, pixels, channels =====
BEAM_FWHM   = 10.0 * U.arcsec
NOISE_RMS   = 0.5e-3 * U.Jy / U.beam     # 0.5 mJy/beam

# pixel = beam/3 (at least 3 pixels per beam FWHM)
px_size = (BEAM_FWHM / 3.0).to(U.arcsec)

# field of view: 1.1× the source extent, but at least 3 beam FWHMs
fov_y = max(y_extent_arcsec, 3 * BEAM_FWHM.to_value(U.arcsec))
fov_z = max(z_extent_arcsec, 3 * BEAM_FWHM.to_value(U.arcsec))

n_px_x = int(np.ceil(fov_y / px_size.value));  n_px_x += n_px_x % 2
n_px_y = int(np.ceil(fov_z / px_size.value));  n_px_y += n_px_y % 2

# built only to estimate the line width; **do not** reuse it for several Martinis (see section 8)
_sm_for_width = GaussianSpectrum(sigma='thermal')

# spectral axis: the particle velocity range **plus** the width of the line itself.
# Before inserting, Martini prunes particles that "cannot fall inside the cube", and one of
# its criteria is that each particle's line ±4σ must lie within the channel range (σ comes
# from GaussianSpectrum). Thermal broadening σ can reach ~100 km/s (gas at tens of millions
# of K), so the spectral axis must be much wider than the gas kinematics.
hw_max = float(np.max(_sm_for_width.half_width(source)).to_value(U.km / U.s))
v_span_needed = v_kms + 8.0 * hw_max            # ±4σ on both sides

channel_width = max(v_span_needed / 40.0, 5.0) * U.km / U.s
# +2 channels of margin: rounding can leave the total width just barely too small, and the
# particles that get pruned are exactly the ones at the edges
n_channels = int(np.ceil(v_span_needed / channel_width.value)) + 2
n_channels += n_channels % 2

print(f'Beam FWHM     {BEAM_FWHM}')
print(f'Pixel         {px_size:.2f}  ({BEAM_FWHM/px_size:.1f} pixels / beam)')
print(f'Field of view {n_px_x}×{n_px_y} px = {n_px_x*px_size.value:.0f}" × {n_px_y*px_size.value:.0f}"')
print(f'              source angular size {y_extent_arcsec:.0f}" × {z_extent_arcsec:.0f}"')
print(f'Spectral axis particle line-of-sight spread {v_kms:.0f} km/s')
print(f'              largest thermal half-width σ = {hw_max:.0f} km/s  →  need ±4σ = ±{4*hw_max:.0f} km/s')
print(f'              total {v_span_needed:.0f} km/s  →  {n_channels} channels × {channel_width:.1f}')
print(f'Noise         {NOISE_RMS.to(U.mJy/U.beam):.1f}')


### 5.2 Beam, noise, spectral model, SPH kernel

| Component | Used in this tutorial | Notes |
|---|---|---|
| `GaussianBeam(bmaj, bmin, bpa, truncate)` | 10″ circular beam | `truncate=4` truncates the kernel at 4σ; `bmin` defaults to `bmaj` |
| `GaussianNoise(rms, seed)` | 0.5 mJy/beam | given a `seed`, the result is reproducible |
| `GaussianSpectrum(sigma)` | `'thermal'` | sets the line width from the particle temperature; a fixed value such as `7*U.km/U.s` also works |
| `CubicSplineKernel()` | cubic spline | MARTINI's SPH kernel; it falls back to a Gaussian kernel automatically when pixels are too coarse |

`GaussianSpectrum(sigma='thermal')` is the **physically correct choice**: the width of the
21 cm line comes from the gas temperature (thermal broadening), and using each particle's
temperature from the simulation is better justified than giving a fixed value.

### 5.3 Running the full pipeline

```python
M = Martini(source=source, datacube=dc, beam=beam, noise=noise,
            spectral_model=..., sph_kernel=...)
M.insert_source_in_cube(ncpu=4)   # paint the spectra, parallel over pixels
M.add_noise()                     # optional
M.convolve_beam()                 # optional
M.write_fits('out.fits')          # or write_hdf5
```

The order of the three steps cannot be changed: **insert the source first, then add noise,
then convolve with the beam**. Adding noise before convolving would smooth the noise as well,
so it would no longer be "per beam" noise.

The next cell **also writes a cube without noise**, after inserting the source and before
adding noise, purely to verify mass conservation (see 5.4) — with noise added, the
single-channel signal-to-noise ratio for this galaxy is far below 1, and summing over the
noisy cube would be dominated by noise. Note that `write_fits()` removes the padding used for
the convolution, so it must be restored afterwards, otherwise the next step,
`convolve_beam()`, raises an error.

In [ ]:
datacube = DataCube(
    n_px_x=n_px_x, n_px_y=n_px_y, n_channels=n_channels,
    px_size=px_size,
    channel_width=channel_width,
    spectral_centre=source.vsys,
    ra=source.ra, dec=source.dec,
    spectral_unit='MHz',          # frequency axis; None gives a velocity axis (see 5.1)
)

beam  = GaussianBeam(bmaj=BEAM_FWHM, bmin=BEAM_FWHM, bpa=0*U.deg, truncate=4.0)
noise = GaussianNoise(rms=NOISE_RMS, seed=42)

M = Martini(source=source, datacube=datacube, beam=beam, noise=noise,
            spectral_model=GaussianSpectrum(sigma='thermal'),   # a fresh instance per Martini
            sph_kernel=CubicSplineKernel())

print('Inserting the source ...')
M.insert_source_in_cube(progressbar=False, ncpu=4)   # this line prints the mass recovery

# Save a cube with **no noise and no convolution** first.
# Mass conservation can only be verified on this one: with noise added the single-channel
# signal-to-noise ratio can be far below 1 (which is the case for this galaxy), and the sum
# would be dominated by noise.
fits_clean = os.path.join(OUTDIR, f'sub{subID}_z{Z_OBS}_10arcsec_nonoise.fits')
M.write_fits(fits_clean)
print(f'Saved (clean) {fits_clean}')

# write_fits() calls datacube.drop_pad() to remove the edge padding used by the convolution,
# so it has to be restored afterwards, otherwise the next step, convolve_beam(), raises
# "datacube padding insufficient".
# (The padding is there so that the beam convolution does not wrap flux around from the
# opposite edge of the image.)
M.datacube.add_pad(M.beam.needs_pad())

print('\nAdding noise ...')
M.add_noise()
print('Convolving with the beam ...')
M.convolve_beam()

fits_hi = os.path.join(OUTDIR, f'sub{subID}_z{Z_OBS}_10arcsec.fits')
M.write_fits(fits_hi)
print(f'Saved {fits_hi}')

### 5.4 Verification: is the mass in the cube right?

The `[xx% of initial source mass]` printed by `insert_source_in_cube()` above is the single
most important diagnostic. **It should be 100%.**

Before inserting, `Martini` prunes particles that "could not contribute to the cube"
(`Martini._prune_particles`). There are three criteria, all enabled:

| Criterion | Meaning |
|---|---|
| **Spatial** | the particle (together with the size of its SPH kernel) falls outside the pixel range of the field of view |
| **Spectral axis** | the particle's line (velocity **±4σ**, with σ from `GaussianSpectrum`) falls outside the channel range |
| **Mass** | `mHI = 0` |

So "a big enough field of view" is equally true of the spectral axis, and **that criterion is
much wider than intuition suggests**: gas at tens of millions of K has a thermal broadening σ
of order 100 km/s, so its line occupies more than ±400 km/s. The
`v_span_needed = v_km + 8σ` of the previous step was computed from exactly this criterion.

If the recovery is below 100%, look at the printed `Pruned particles that will not contribute
to data cube, N particles remaining with total HI mass of ...` — how much the total has
dropped is how much was pruned. A wider field of view / channel range fixes it.

The next cell **does not rely on any internal variable of astrovodka**: it recovers the mass
straight from the written FITS file, which is exactly what an observer does with a real data
cube. (It uses the cube from above that has **not yet been convolved** with the beam, in
`Jy/arcsec²`, the most direct conversion; if the cube has been convolved with the beam and its
units have become `Jy/beam`, you must divide by the beam solid angle `π/(4 ln2)·bmaj·bmin` as
well.) Using the standard formula:

$$M_{\rm HI}\ [M_\odot] = 2.36\times10^5 \cdot D_L^2 [\mathrm{Mpc}] \cdot \sum_{\rm chan} S_{\rm tot}\, \Delta v\ [\mathrm{Jy\cdot km/s}]$$

where $S_{\rm tot}$ is the **total** flux density of that channel: multiply each pixel's
`Jy/arcsec²` by the pixel area and sum. All units are read from the FITS header.

In [ ]:
with fits.open(fits_clean) as hdul:
    head = hdul[0].header
    cube = hdul[0].data.astype('float64')

px_arcsec = abs(head['CDELT1']) * 3600.0            # degrees -> arcsec (CDELT1 is negative)

# channel width: read the spectral axis unit from CUNIT3, do not assume
if head['CUNIT3'].strip().replace(' ', '') in ('ms-1', 'm/s'):
    dv_kms = abs(head['CDELT3']) / 1000.0
else:                                                # frequency axis, Hz
    dv_kms = abs(head['CDELT3']) / head['RESTFRQ'] * 299792.458

print(f'FITS header: BUNIT={head["BUNIT"]!r}  CTYPE3={head["CTYPE3"]}  '
      f'CUNIT3={head["CUNIT3"]!r}')
print(f'  pixel {px_arcsec:.3f}"   {cube.shape[0]} channels × {dv_kms:.3f} km/s')

# this one has not been convolved with the beam, so its units are Jy/arcsec^2, the most
# direct conversion:
# total flux density per channel [Jy] = Σ_pix S[Jy/arcsec^2] × pixel area
s_tot = cube.sum(axis=(1, 2)) * px_arcsec**2          # Jy
integrated = float(np.sum(s_tot * dv_kms))            # Jy km/s

DL_mpc = luminosity_distance(Z_OBS, cosmo).value
m_hi_fits = 2.36e5 * DL_mpc**2 * integrated           # Msun

print(f'\nIntegrated flux ∫S dv = {integrated:.6e} Jy km/s')
print(f'M_HI (recovered from the FITS)  = {m_hi_fits:.4e}')
print(f'M_HI (given directly by the model)  = {source.m_HI_total:.4e}')
ratio = m_hi_fits / source.m_HI_total.value
print(f'ratio = {ratio:.4f}')
print('\n→ It should be close to 1.00.')
print('  The missing fraction of a percent: part of the SPH kernel of edge particles is\n'
      '  deposited outside the grid, and that flux really is gone — an inherent loss of a')
print('  finite field of view, not a bug.')
print('  Only a value clearly below ~0.99 means whole particles were pruned, in which case\n'
      '  enlarge the field of view or the channel range.')
assert 0.99 < ratio < 1.01, f'anomalous mass recovery: {ratio}'

# if the cube has already been convolved with the beam (units Jy/beam), divide by the beam
# solid angle as well:
#   S_total[Jy] = Σ_pix S[Jy/beam] × px_area / beam_area
#   beam_area = pi/(4 ln2) × bmaj × bmin
# This one is not convolved, so that does not apply here; but watch BUNIT when reading a
# cube that someone else gives you.

# ---- is the noise as advertised? ----
with fits.open(fits_hi) as hdul:
    noisy = hdul[0].data.astype('float64')
rms_meas = float(np.std(noisy[0]))
print(f'\nNoise: requested {NOISE_RMS.to(U.mJy/U.beam):.2f} mJy/beam, '
      f'measured in a single channel of the final cube {rms_meas*1e3:.2f} mJy/beam')
peak_chan = float(np.max(noisy[0]))
print(f'  single-channel peak {peak_chan*1e3:.2f} mJy/beam  →  single-channel S/N '
      f'{peak_chan/rms_meas:.2f}')
print('  (at this noise level the galaxy is undetectable channel by channel; you have to\n'
      '  integrate over moment 0 to see it)')

### 5.5 Changing the angular resolution

The same source and the same noise, with the beam changed from 10″ to 3′.
This is the "high-resolution interferometer" vs "single dish" comparison: at low resolution
each beam holds more flux and the single-channel signal-to-noise ratio is higher, but the
spatial information is washed out.

In [ ]:
BEAM_B = 3.0 * U.arcmin
px_size_b = (BEAM_B / 3.0).to(U.arcsec)

fov_y_b = max(y_extent_arcsec, 3 * BEAM_B.to_value(U.arcsec))
fov_z_b = max(z_extent_arcsec, 3 * BEAM_B.to_value(U.arcsec))
n_px_x_b = int(np.ceil(fov_y_b / px_size_b.value));  n_px_x_b += n_px_x_b % 2
n_px_y_b = int(np.ceil(fov_z_b / px_size_b.value));  n_px_y_b += n_px_y_b % 2

datacube_b = DataCube(
    n_px_x=n_px_x_b, n_px_y=n_px_y_b, n_channels=n_channels,
    px_size=px_size_b, channel_width=channel_width,
    spectral_centre=source.vsys, ra=source.ra, dec=source.dec,
    spectral_unit='MHz',
)
print(f'Beam {BEAM_B:.0f}, pixel {px_size_b:.0f}", '
      f'field of view {n_px_x_b}×{n_px_y_b} px = {n_px_x_b*px_size_b.value:.0f}" × '
      f'{n_px_y_b*px_size_b.value:.0f}"')

M_b = Martini(source=source, datacube=datacube_b,
              beam=GaussianBeam(bmaj=BEAM_B, bmin=BEAM_B, bpa=0*U.deg, truncate=4.0),
              noise=GaussianNoise(rms=NOISE_RMS, seed=42),
              spectral_model=GaussianSpectrum(sigma='thermal'),
              sph_kernel=CubicSplineKernel())

M_b.insert_source_in_cube(progressbar=False, ncpu=4)
M_b.add_noise()
M_b.convolve_beam()

fits_lo = os.path.join(OUTDIR, f'sub{subID}_z{Z_OBS}_3arcmin.fits')
M_b.write_fits(fits_lo)
print(f'\nSaved {fits_lo}')

---

## 6. 3D data products

`SubhaloDataProducts` is not a data cube but a **3D grid** — suited to statistics, power
spectra, or feeding another radiative transfer code.

| Grid | Units | Meaning |
|---|---|---|
| `uv_field` | Draine | UV radiation field |
| `dm_real` / `dm_rsd` | M☉/(ckpc/h)³ | dark matter density (real space / redshift space) |
| `hi_real` / `hi_rsd` | M☉/(ckpc/h)³ | HI density |
| `h2_real` / `h2_rsd` | M☉/(ckpc/h)³ | H₂ density |

The redshift-space displacement uses `s_x = x + v_los/(a·H(a))`, which is what an observer
actually sees.

> **Two coordinate frames, do not mix them.** All the mass grids are in **comoving ckpc/h**,
> in a box centred on the DM centre of mass (`dp.grid_edges`). But `uv_field` lives in an
> adaptive box in **physical kpc** (`dp.metadata['uv_grid_bounds_physical_kpc']`) — it follows
> the gas distribution. The two are not the same.

**Section 6.3 goes to group scale**: `GroupDataProducts` grids the whole FOF group the
subhalo belongs to, and splits its gas into ISM, CGM and IGM.

In [ ]:
print(f'Computing the 3D data products (ngrid=128, model {HI_MODEL}) ...')
dp = SubhaloDataProducts.from_subhalo(
    basePath=basePath, snapNum=snapNum, subID=subID,
    ngrid=128, hi_model=HI_MODEL,
    load_subhalo_only=True, cosmology=cosmo,
)

h5_path = os.path.join(OUTDIR, f'sub{subID}_z{Z_OBS}_data_products.hdf5')
dp.save_hdf5(h5_path)
print(f'Saved {h5_path}')

print(f'\nGrid {dp.dm_real.shape}, units {dp.metadata["grid_unit"]}')
print(f'  total M_HI {dp.metadata["m_HI_Msun"]:.4e}  (should equal source.m_HI_total)')
print(f'  total M_H2 {dp.metadata["m_H2_Msun"]:.4e}')
print(f'  model used {dp.metadata["hi_model"]}')
print(f'  UV field   {"none (projected model)" if dp.uv_field is None else dp.uv_field.shape}')
if dp.metadata.get('uv_grid_bounds_physical_kpc'):
    b = dp.metadata['uv_grid_bounds_physical_kpc']
    print(f'    UV box [physical kpc] x=[{b[0]:.0f},{b[1]:.0f}] y=[{b[2]:.0f},{b[3]:.0f}] z=[{b[4]:.0f},{b[5]:.0f}]')

# ---- round trip test ----
dp2 = SubhaloDataProducts.load_hdf5(h5_path)
assert np.array_equal(dp.hi_real, dp2.hi_real)
print('\n✓ HDF5 round trip consistent')

### 6.1 Density projections

In [ ]:
def plot_projections(dp, fields, titles, cmaps, figname):
    """Project the 3D grid along the three axes, labelling the axes with physical scales."""
    nrows = len(fields)
    fig, axes = plt.subplots(nrows, 3, figsize=(15, 4.3 * nrows))
    axes = np.atleast_2d(axes)
    ext = [dp.grid_edges[0][0], dp.grid_edges[0][-1],
           dp.grid_edges[1][0], dp.grid_edges[1][-1]]
    for r, (f, t, cm) in enumerate(zip(fields, titles, cmaps)):
        arr = getattr(dp, f)
        for c, lbl in enumerate(['xy', 'xz', 'yz']):
            proj = arr.sum(axis=2 - c)
            ax = axes[r, c]
            pos = proj[proj > 0]
            im = ax.imshow(proj.T, origin='lower', extent=ext, cmap=cm,
                           norm=LogNorm(vmin=pos.min() if pos.size else 1,
                                        vmax=proj.max()))
            ax.set_title(f'{t} — {lbl}', fontsize=11)
            ax.set_xlabel('ckpc/h'); ax.set_ylabel('ckpc/h')
            plt.colorbar(im, ax=ax, shrink=.85)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, figname), bbox_inches='tight', facecolor='white')
    plt.show()


plot_projections(
    dp,
    ['dm_real', 'hi_real', 'h2_real'],
    ['DM density', 'HI density', r'H$_2$ density'],
    ['bone', 'inferno', 'magma'],
    f'sub{subID}_z{Z_OBS}_DM_HI_projections.png',
)

### 6.2 The UV field and redshift-space distortions

`dm_rsd − dm_real` is the effect of redshift-space distortions (RSD) — peculiar velocities
stretch structure along the line of sight (the Kaiser effect) or squash it (the fingers of
God).

In [ ]:
def plot_uv_and_rsd(dp, figname):
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
    ex, ey, ez = dp.grid_edges

    if dp.uv_field is not None:
        ax = axes[0]
        b = dp.metadata.get('uv_grid_bounds_physical_kpc')
        ext = [b[0], b[1], b[2], b[3]] if b else None
        im = ax.imshow(dp.uv_field.sum(axis=2).T, origin='lower', extent=ext,
                       cmap='afmhot', norm=LogNorm())
        ax.set_title('UV field (summed along z, physical kpc)')
        ax.set_xlabel('kpc'); ax.set_ylabel('kpc')
        # note: this plots the **sum** along one line of sight, not the U_MW of a single cell
        plt.colorbar(im, ax=ax, label=r'$\sum$ U_MW [Draine]', shrink=.85)
    else:
        axes[0].text(.5, .5, 'the projected model produces no 3D UV field grid\n(its UV field lives in the projection plane)',
                     ha='center', va='center', transform=axes[0].transAxes)
        axes[0].set_axis_off()

    ax = axes[1]
    diff = dp.dm_rsd - dp.dm_real
    v = np.percentile(np.abs(diff[diff != 0]), 99) if np.any(diff != 0) else 1
    im = ax.imshow(diff.sum(axis=2).T, origin='lower',
                   extent=[ex[0], ex[-1], ey[0], ey[-1]],
                   cmap='RdBu_r', vmin=-v, vmax=v)
    ax.set_title('redshift-space distortions  DM_RSD − DM_real')
    ax.set_xlabel('ckpc/h'); ax.set_ylabel('ckpc/h')
    plt.colorbar(im, ax=ax, shrink=.85)

    ax = axes[2]
    k = dp.hi_real.shape[2] // 2
    im = ax.imshow(dp.hi_real[:, :, k].T, origin='lower',
                   extent=[ex[0], ex[-1], ey[0], ey[-1]],
                   cmap='inferno', norm=LogNorm())
    ax.set_title(f'HI density mid-plane slice (z index {k})')
    ax.set_xlabel('ckpc/h'); ax.set_ylabel('ckpc/h')
    plt.colorbar(im, ax=ax, shrink=.85)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, figname), bbox_inches='tight', facecolor='white')
    plt.show()


plot_uv_and_rsd(dp, f'sub{subID}_z{Z_OBS}_UV_and_RSD.png')

### 6.3 Zooming out: the FOF group

Every subhalo knows which group it lives in — `SubhaloGrNr`. Ours is not a lucky one:

```python
loadSingle(basePath, snapNum, subhaloID=subID)['SubhaloGrNr']    # -> 0
```

Group 0 is the most massive object in the box: **17,185 subhalos and 4.5e7 gas cells**
within R200c = 1.5 Mpc. Loading that is several GB of reads, so this section uses a
smaller group instead — the machinery is identical.

`GroupDataProducts.from_group()` grids a whole **FOF group**, every subhalo in it plus
whatever belongs to the group alone, and splits the gas by where it physically is:

| Class | Definition |
|---|---|
| **ISM** | bound to a subhalo and within `min(k_gas · R_half,gas, k_star · R_half,star)` of its centre |
| **CGM** | bound to a subhalo but outside that radius |
| **IGM** | in the group but bound to **no** subhalo — the group's "fuzz" |

Membership is not a spatial guess. The catalog stores, for every subhalo, the range of
gas cells it owns in the snapshot's particle array (`Subhalo/SnapByType` +
`SubhaloLenType`), so the split is exact and costs nothing extra to compute.

Three things are worth knowing before you point this at your own group:

- **The ISM radius has to be the minimum of two radii.** A group's central subhalo
  contains the whole hot halo, so its *gas* half-mass radius is ~260 ckpc/h against
  ~19 ckpc/h for the *stellar* one (group 59). A gas-only radius at `k=0.5` would
  already call 15% of the group's mass ISM.
- **HI is computed per subhalo**, each in its own adaptive FFT box — the prescription
  the models were calibrated under. Doing it once for the whole group would smear the
  UV field over ~30 kpc cells. Only gas-bearing subhalos are modelled: 68 of group
  59's 1710 members.
- **The class fractions do not depend on `ngrid`** — they are per-cell quantities. The
  grid only decides how the answer is displayed.

In [ ]:
# Where does our subhalo actually live, and what would its group cost?
# One catalog row each -- this is cheap, so look before you load.
h = loadHeader(basePath, snapNum)['HubbleParam']
for gid, label in ((int(loadSingle(basePath, snapNum, subhaloID=subID)['SubhaloGrNr']),
                    "our subhalo's group"), (GROUP_ID, 'the group used below')):
    g = loadSingle(basePath, snapNum, haloID=gid)
    print(f"{label:<22} group {gid:>5}: {int(g['GroupNsubs']):>6} subhalos, "
          f"{int(g['GroupLenType'][0]):>12,} gas cells, "
          f"{int(g['GroupLenType'][1]):>12,} DM particles, "
          f"M200c = {g['Group_M_Crit200'] * 1e10 / h:.2e} Msun")

import time
t0 = time.time()
gdp = GroupDataProducts.from_group(
    basePath, snapNum, GROUP_ID,
    ngrid=GROUP_NGRID,
    hi_model=HI_MODEL,          # only the volumetric models apply to a group
    ncpu=1,
)
print(f"\nloaded group {GROUP_ID} in {time.time() - t0:.0f} s")

md = gdp.metadata
print(f"{md['Nsubs']} subhalos ({md['N_sub_with_gas']} with gas), {md['n_gas']:,} gas cells, "
      f"R200c = {md['R200c_kpc']:.0f} kpc, M200c = {md['M200c_msun']:.2e} Msun")
print(f"box half-width [los, y, z] = "
      f"{[round(float(v), 1) for v in md['box_halfsize_ckpc']]} ckpc/h")

print(f"\n{'class':<6}{'cells':>11}{'gas mass [Msun]':>18}{'HI mass [Msun]':>17}"
      f"{'f_HI':>8}{'gas frac':>10}")
for c in ('ism', 'cgm', 'igm'):
    gas, hi = md[f'm_gas_{c}_msun'], md[f'm_hi_{c}_msun']
    print(f"{c.upper():<6}{md['n_cells_' + c]:>11,}{gas:>18.4e}{hi:>17.4e}"
          f"{hi / max(gas, 1):>8.4f}{100 * gas / md['m_gas_total_msun']:>9.2f}%")

# The three classes partition the group exactly, by construction...
assert md['n_cells_ism'] + md['n_cells_cgm'] + md['n_cells_igm'] == md['n_gas']
assert abs((md['m_gas_ism_msun'] + md['m_gas_cgm_msun'] + md['m_gas_igm_msun'])
           / md['m_gas_total_msun'] - 1) < 1e-9
# ... and the classification means something: the ISM is where the atomic gas is.
f_hi = {c: md[f'm_hi_{c}_msun'] / md[f'm_gas_{c}_msun'] for c in ('ism', 'cgm', 'igm')}
assert f_hi['ism'] > f_hi['cgm'] > f_hi['igm'], f_hi
print(f"\n\u2713 classes partition the group exactly, and f_HI(ISM)={f_hi['ism']:.3f} "
      f"> f_HI(CGM)={f_hi['cgm']:.3f} > f_HI(IGM)={f_hi['igm']:.4f}")
print(f"  the fuzz is {100 * md['m_gas_igm_msun'] / md['m_gas_total_msun']:.1f}% of the gas "
      f"and {100 * md['m_dm_igm_total_msun'] / md['m_dm_total_msun']:.1f}% of the dark matter")

In [ ]:
def plot_group_classes(dp, figname):
    """The three classes across the group, then the brightest galaxy close up."""
    s = dp.metadata['hubble_h'] * dp.metadata['scale_factor']   # physical kpc per ckpc/h
    e = [np.asarray(x) / s for x in dp.grid_edges]
    centre = np.asarray(dp.metadata['group_pos_ckpc_h']) / s
    r_ism = ism_radius(dp.subhalos['rhalf_type'][:, 0], dp.subhalos['rhalf_type'][:, 4],
                       dp.metadata['ism_k_gas'], dp.metadata['ism_k_star'])

    fig, axes = plt.subplots(2, 3, figsize=(16, 8.8))

    # ---- top row: the whole group, class by class ----
    for c, cls in enumerate(('ism', 'cgm', 'igm')):
        ax = axes[0, c]
        img = getattr(dp, f'gas_{cls}_real').sum(axis=2).T      # rows: y, columns: x
        pos = img[img > 0]
        im = ax.imshow(img, origin='lower', aspect='auto',
                       extent=[e[0][0] - centre[0], e[0][-1] - centre[0],
                               e[1][0] - centre[1], e[1][-1] - centre[1]],
                       cmap='magma',
                       norm=LogNorm(vmin=pos.min() if pos.size else 1e-6,
                                    vmax=np.percentile(pos, 99.5) if pos.size else 1.0))
        ax.add_patch(Circle((0, 0), dp.metadata['R200c_ckpc_h'] / s, fill=False,
                            ec='cyan', ls='--', lw=1.2, label='R200c'))
        frac = 100 * dp.metadata[f'm_gas_{cls}_msun'] / dp.metadata['m_gas_total_msun']
        ax.set_title(f'{cls.upper()} — {frac:.1f}% of the gas')
        ax.set_xlabel(r'$\Delta$x [kpc]'); ax.set_ylabel(r'$\Delta$y [kpc]')
        plt.colorbar(im, ax=ax, shrink=.85, label=r'$M_\odot$/kpc$^3$')

    # ---- bottom row: the brightest galaxy, re-gridded from the per-cell data ----
    # A group-wide grid is ~30 kpc per cell, so zooming into it would show blocks;
    # dp.cells lets us histogram just this sub-volume at a galactic scale instead.
    xyz = np.asarray(dp.cells['coords_ckpc_h'], dtype='float64')
    mass = np.asarray(dp.cells['m_gas_msun'], dtype='float64')
    klass = np.asarray(dp.cells['class_id'])
    gal = int(np.argmax(np.asarray(dp.subhalos['mass_type'])[:, 0]))   # most gas
    gc = np.asarray(dp.subhalos['pos'])[gal]
    half = float(np.clip(4.0 * r_ism[gal], 60.0 * s, 0.3 * dp.metadata['R200c_ckpc_h']))
    edges = [np.linspace(gc[i] - half, gc[i] + half, 257) for i in range(2)]
    inside = np.all(np.abs(xyz - gc) < half, axis=1)
    area = (edges[0][1] - edges[0][0]) * (edges[1][1] - edges[1][0])
    ext = [edges[0][0] / s - gc[0] / s, edges[0][-1] / s - gc[0] / s,
           edges[1][0] / s - gc[1] / s, edges[1][-1] / s - gc[1] / s]
    maps, ranges = {}, {}
    for cls, cid in (('ISM', 0), ('CGM', 1)):
        sel = inside & (klass == cid)
        maps[cls] = np.histogram2d(xyz[sel, 0], xyz[sel, 1], bins=edges,
                                   weights=mass[sel])[0] / area
        pos = maps[cls][maps[cls] > 0]
        ranges[cls] = (np.percentile(pos, 25) if pos.size else 1e-6,
                       np.percentile(pos, 99.5) if pos.size else 1.0)
    for c, key in enumerate(('ISM', 'CGM', None)):
        ax = axes[1, c]
        if key is None:
            img = np.zeros(maps['ISM'].T.shape + (3,))
            for cls, rgb in (('ISM', 0), ('CGM', 2)):
                lo, hi = ranges[cls]
                img[..., rgb] = np.clip(
                    (np.log10(np.maximum(maps[cls].T, lo)) - np.log10(lo))
                    / max(np.log10(hi) - np.log10(lo), 1e-9), 0, 1)
            ax.imshow(img, origin='lower', extent=ext)
            ax.set_title('both (red = ISM, blue = CGM)')
        else:
            lo, hi = ranges[key]
            im = ax.imshow(maps[key].T, origin='lower', extent=ext, cmap='magma',
                           norm=LogNorm(vmin=lo, vmax=hi))
            ax.set_title(f'{key}: brightest galaxy')
            plt.colorbar(im, ax=ax, shrink=.85, label=r'$M_\odot$/kpc$^2$')
        ax.set_xlabel(r'$\Delta$x [kpc]'); ax.set_ylabel(r'$\Delta$y [kpc]')

    fig.suptitle(f"group {dp.metadata['groupID']}: ISM inside the galaxies, CGM in their "
                 f"haloes, IGM where nothing is bound (bottom row: ±{half / s:.0f} kpc)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, figname), bbox_inches='tight', facecolor='white')
    plt.show()


plot_group_classes(gdp, f'group{GROUP_ID:07d}_snap{snapNum:03d}_ism_cgm_igm.png')

---

## 7. Visualizing the data cube

A handy function: read the cube back from the FITS file and plot moment 0/1, the global
spectrum and a few channel maps. This is the standard way to check "what did I actually
make".

In [ ]:
def compute_moments(cube, snr=3.0):
    """Moment 0 and moment 1.

    Moment 1 is computed only on pixels **with signal** — otherwise the noise would be
    divided by a moment 0 close to zero, giving garbage values of tens of thousands of
    km/s. Pixels without signal are set to NaN.
    """
    mom0 = cube.sum(axis=0)
    noise = np.std(cube[0])                              # single-channel noise
    # the moment-0 noise is estimated by error propagation: summing n independent channels
    # gives a factor sqrt(n). (Not measured from the image edges: the field of view here is
    # only 1.1× the source, so there is source at the edges too and no clean region.)
    mom0_noise = noise * np.sqrt(cube.shape[0])
    sig = mom0 > snr * mom0_noise
    with np.errstate(invalid='ignore', divide='ignore'):
        mom1 = (cube * np.arange(cube.shape[0])[:, None, None]).sum(axis=0) / mom0
    mom1 = np.where(sig, mom1, np.nan)
    return mom0, mom1, noise, sig


def visualize_cube(fits_path, title, figname, n_chan_maps=5):
    with fits.open(fits_path) as hdul:
        cube = hdul[0].data.astype('float64')
        head = hdul[0].header
    mom0, mom1, noise, sig = compute_moments(cube)
    nchan = cube.shape[0]
    # the source's spectral centre is at the centre of the cube (the datacube is centred on source.vsys)
    centre_chan = nchan // 2
    mom0_noise = noise * np.sqrt(nchan)
    peak_snr = mom0.max() / mom0_noise

    fig = plt.figure(figsize=(16, 8.6))
    gs_top = fig.add_gridspec(1, 3, top=.86, bottom=.42, wspace=.34)
    gs_bot = fig.add_gridspec(1, n_chan_maps, top=.32, bottom=.04, wspace=.28)
    fig.subplots_adjust(left=.06, right=.97)

    ax = fig.add_subplot(gs_top[0, 0])
    vmax = np.percentile(mom0[sig], 98) if sig.any() else mom0.max()
    im = ax.imshow(mom0, origin='lower', cmap='inferno', vmin=0, vmax=vmax)
    ax.set_title('Moment 0 (integrated flux)', fontweight='bold')
    ax.contour(sig, levels=[.5], colors='w', linewidths=.6, alpha=.6)
    plt.colorbar(im, ax=ax, label='Jy/beam · channel')

    ax = fig.add_subplot(gs_top[0, 1])
    if np.any(np.isfinite(mom1)):
        lo, hi = np.nanpercentile(mom1, [2, 98])
        im = ax.imshow(mom1, origin='lower', cmap='RdBu_r', vmin=lo, vmax=hi)
        plt.colorbar(im, ax=ax, label='channel index')
    else:
        ax.text(.5, .5, 'signal too weak to compute a velocity field', ha='center', va='center',
                transform=ax.transAxes)
    ax.set_title('Moment 1 (velocity field, signal pixels only)', fontweight='bold')

    ax = fig.add_subplot(gs_top[0, 2])
    spec = cube.sum(axis=(1, 2))
    cnoise = noise * np.sqrt(cube.shape[1] * cube.shape[2])
    ax.plot(spec, np.arange(nchan), 'k-', lw=1.4)
    ax.axvspan(-cnoise, cnoise, color='C0', alpha=.18, label=r'±1σ (summed over the image)')
    ax.axhline(centre_chan, color='r', ls='--', lw=1, label='spectral centre')
    ax.invert_yaxis()
    ax.set_xlabel('total flux [Jy/beam]'); ax.set_ylabel('channel')
    ax.set_title('global spectrum', fontweight='bold')
    ax.legend(fontsize=8)

    chans = np.unique(np.clip(
        np.linspace(centre_chan - n_chan_maps // 2,
                    centre_chan + n_chan_maps // 2, n_chan_maps),
        0, nchan - 1).astype(int))
    for i, ch in enumerate(chans):
        ax = fig.add_subplot(gs_bot[0, i])
        ax.imshow(cube[ch], origin='lower', cmap='inferno',
                  vmin=0, vmax=np.percentile(cube[ch], 99.5))
        ax.set_title(f'channel {ch}', fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(title, fontsize=15, fontweight='bold', y=.96)
    plt.savefig(os.path.join(OUTDIR, figname), bbox_inches='tight', facecolor='white')
    plt.show()

    print(f'{os.path.basename(fits_path)}')
    print(f'  single-channel noise {noise*1e3:.2f} mJy/beam')
    print(f'  moment-0 noise (propagated) {mom0_noise*1e3:.3f} mJy/beam·channel')
    print(f'  moment-0 peak/noise = {peak_snr:.1f}')
    print(f'  pixels above 3σ {int(sig.sum())} / {sig.size}')
    print(f'  Note: this is a **per-pixel** signal-to-noise ratio. The source is extended, so')
    print(f'  integrating the whole of it gives a much higher significance — which is why you\n'
          f'  want to look at moment 0 rather than single-channel maps.')


visualize_cube(fits_hi, f'TNG-100 sub{subID}, z={Z_OBS}, 10 arcsec beam',
               f'sub{subID}_z{Z_OBS}_10arcsec_viz.png')
visualize_cube(fits_lo, f"TNG-100 sub{subID}, z={Z_OBS}, 3 arcmin beam",
               f'sub{subID}_z{Z_OBS}_3arcmin_viz.png')

### 7.1 The two resolutions side by side

The same flux, spread out by beams of different size. At low resolution each beam holds more
flux and the single-channel signal-to-noise ratio is better, but structure is washed out —
which is exactly why HI observations have to make this trade-off.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, fpath, label in zip(axes, [fits_hi, fits_lo],
                            ['10 arcsec beam', "3 arcmin beam"]):
    with fits.open(fpath) as hdul:
        d = hdul[0].data.astype('float64')
    mom0 = d.sum(axis=0)
    rms = np.std(d[0])
    sig = mom0 > 3 * rms
    vmax = np.percentile(mom0[sig], 98) if sig.any() else mom0.max()
    im = ax.imshow(mom0, origin='lower', cmap='inferno', vmin=0, vmax=vmax)
    peak_sig = mom0.max() / np.std(mom0[~sig])
    ax.set_title(f'{label}   (moment-0 peak/noise = {peak_sig:.1f})',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('RA pixel'); ax.set_ylabel('Dec pixel')
    plt.colorbar(im, ax=ax, label='Jy/beam · channel', shrink=.85)
plt.suptitle(f'TNG-100 sub{subID} @ z={Z_OBS} — Moment 0', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, f'sub{subID}_z{Z_OBS}_comparison.png'),
            bbox_inches='tight', facecolor='white')
plt.show()

---

## 8. A trap you must know: `Martini` prunes the source **in place**

`Martini._prune_particles` deletes particles that "cannot fall inside the cube" from the
**`source` object itself** (`source.apply_mask`); it does not merely ignore them for this one
observation. The consequence is:

> **Build two cubes of different sizes from the same `source`, and the second one comes out
> short.**

The first cube's field of view is small → a ring of particles is pruned → `source.npart` gets
smaller → the second cube receives an already incomplete source. And the loss is **silent**;
it shows up only in the recovery fraction printed by `insert_source_in_cube`.

The right way to do it:

- **one source per observation**; for a parameter sweep, construct a fresh `TNGSource` for
  each set of parameters;
- or make sure the first cube's field of view / spectral axis already holds the whole source
  (100% recovery), in which case no particles are pruned and the source is unaffected.

The check below demonstrates this: it runs **after all the science products have been made**,
and still on the same `source` — because those three small cubes are guaranteed to prune the
source.

In [ ]:
# ===== verify the spectral axis unit: km/s and MHz must give the same cube =====
# Keep this at the end! The field of view of the small cubes below cannot hold the whole
# galaxy, so they prune `source` in place. If this ran earlier, every later cube would be
# missing a chunk of the source.
print('Comparing the two spectral axes with a 24×24×16 small cube (only to compare the flux; the field of view is deliberately too small)\n')

flux = {}
for unit in ('km/s', 'MHz'):
    dc_small = DataCube(n_px_x=24, n_px_y=24, n_channels=16,
                        px_size=px_size, channel_width=channel_width,
                        spectral_centre=source.vsys, ra=source.ra, dec=source.dec,
                        spectral_unit=unit)
    m = Martini(source=source, datacube=dc_small, quiet=True,   # quiet is a constructor argument
                spectral_model=GaussianSpectrum(sigma='thermal'),
                sph_kernel=CubicSplineKernel())
    m.insert_source_in_cube(progressbar=False)
    flux[unit] = float(np.sum(m.datacube._array.to_value(U.Jy / U.arcsec**2)))
    print(f'  spectral_unit={unit!r:6}  CTYPE3={dc_small.wcs.wcs.ctype[2]:>4}  '
          f'channel width={dc_small.channel_width}  total cube flux={flux[unit]:.6e}')

ratio = flux['MHz'] / flux['km/s']
print(f'\n  flux ratio MHz/(km/s) = {ratio:.6f}   (1.000000 = consistent)')
assert abs(ratio - 1.0) < 1e-6, 'the spectral axis unit conversion is wrong!'
print('  ✓ the two spectral axes give the same cube.')
print('    In the old version this was 0.00007 — it converted channel_width with')
print('    doppler_radio, which returns f_rest itself rather than a width, leaving only noise.')
print(f'\n  after this run source.npart = {source.npart} (pruned out of the {source.n_gas} gas cells)')

---

## 9. Troubleshooting

### Errors

| Error | Cause / what to do |
|---|---|
| `ValueError: unknown hi_model 'K13_volumn'` | Misspelled name. The valid values are `GD14_vol`, `GD14_map`, `K13_vol`, `K13_map` (case-insensitive). Validation happens before anything is read from disk, so a typo does not cost you a long wait. |
| `ValueError: uv_mode='global' does not apply to K13_map` | The projected models get their UV field from the SFR surface-density map, which is part of the model definition. Drop `uv_mode`, or switch to a `*_vol` model. |
| `ValueError: the K13_map model projects the galaxy, so it needs the stellar particle data` | When calling `compute_hi_mass` directly, the projected models need `stars=` and `sub=`. With `TNGSource` you will not hit this. |
| `KeyError: 'Coordinates' ...` | The gas dict passed to `compute_hi_mass` is missing a field. The volumetric models need 8 fields; the projected ones do not need `InternalEnergy`, but they do need stars. |
| `UserWarning: ... angular diameter distance of only 4.4 kpc` | Snapshot 99 has z≈0 and `observed_redshift` was not passed. See 3.2. |
| `TypeError: unknown HIModelConfig field(s) ['n_grid']` | A field name in `hi_config` is misspelled; the error lists the available names. |
| The second cube's recovery fraction drops for no apparent reason | **`Martini` prunes `source` in place.** The previous cube's field of view / spectral axis was too small and deleted particles from the source. See section 8: one source per observation, or make sure the first cube is already at 100%. |
| `ValueError: datacube padding insufficient for beam convolution` | `write_fits()`/`write_hdf5()` was called before `convolve_beam()` — they call `drop_pad()`. Either convolve before writing, or restore the padding afterwards: `M.datacube.add_pad(M.beam.needs_pad())`. |
| `ValueError: cannot reshape array of size ... into shape ...` | **The same `GaussianSpectrum` instance is being reused by several `Martini` objects.** The spectrum arrays are cached on the model object, and `Martini` re-initialises them only while they are still `None`, so a change of cube shape (especially the number of channels) makes it apply the old shape. Build a fresh instance for each `Martini`, or call `M.init_spectra()` explicitly. |

### The results look wrong

| Symptom | Most likely cause |
|---|---|
| `insert_source_in_cube` reports a mass recovery << 100% and prints `Pruned particles ...` | **The cube's field of view is too small.** Enlarge `n_px_x/n_px_y` to match the source's angular size (section 5.1). |
| The cube is nothing but noise | The source has too little HI (check `source.m_HI_total`), or the noise `rms` is too large, or the channel range does not cover the source's line-of-sight velocity. |
| The projected model gives less total HI than the volumetric one | Normal: gas outside the projection box is recorded as zero. Look at `source.hi_diagnostics['f_gas_in_map']`; below 0.9, `TNGSource` warns. To keep that gas, set `hi_config={'map_clip_outside': True}`. |
| Loading one subhalo takes minutes and the memory blows up | **You picked up a cluster core.** The catalog is sorted by descending mass, and ID 0 has about 38 million cells. Use the 100~200 range instead. |
| `adaptive_fft` and `global` give nearly the same result | This galaxy does not form many stars, so its UV field peak sits right on the background. It takes a subhalo with active star formation to see the difference. |

### Numerical consistency checklist

It is worth running these three at any time:

1. the recovery fraction of `insert_source_in_cube()` = **100%**
2. the M_HI recovered from the FITS ≈ `source.m_HI_total` (section 5.4)
3. the `m_neutral` of the two volumetric models is **exactly the same** (the models differ only in how the HI/H₂ split is made)

---

## 10. Summary and next steps

### What this walkthrough did

1. **Choosing a subhalo** — avoiding the cluster cores that come first in the mass-sorted order
2. **Loading the source** — `TNGSource` reads the snapshot, computes Planck15 distances, runs the chosen HI model
3. **Comparing the four models** — `compute_hi_mass(gas, model=...)`, four runs on one copy of the data
4. **Building the observation** — `DataCube` (with the spectral axis unit trap) + beam + noise + spectral model + SPH kernel
5. **Verification** — 100% mass recovery, FITS recovery matching the model, the same flux from both spectral axes
6. **3D products** — UV/DM/HI/H₂ grids, in real space and in redshift space
7. **Zooming out to the group** — `GroupDataProducts` splits a whole FOF group into
   ISM / CGM / IGM using the catalog's own membership
8. **Visualization** — moment maps, spectra, channel maps

### Key interfaces at a glance

```python
# four models, pick one
TNGSource(..., hi_model='K13_vol')          # high level: one step
compute_hi_mass(gas, 'K13_map', cfg, stars=stars, sub=sub)   # low level: physics only

# configuration
HIModelConfig(h=0.6774, z=0.0)              # a complete object
HIModelConfig.build({'ngrid': 64}, h=..., z=...)   # merged, with the snapshot taking precedence
TNGSource(..., hi_config={'ngrid': 64})     # a dict works too

# model results
source.mHI_g, source.m_H2_g, source.f_mol_g, source.u_mw_g
source.hi_diagnostics['f_gas_in_map']

# the whole FOF group the subhalo lives in, split by class
gdp = GroupDataProducts.from_group(basePath, snapNum, GROUP_ID, ngrid=192)
gdp.metadata['m_gas_ism_msun'], gdp.metadata['m_hi_total_msun']
gdp.gas_ism_real, gdp.cells['class_id']    # grids, and the per-cell labels
```

### Documentation and verification

| Want to know about | Look at |
|---|---|
| The whole API | [`docs/en/api/`](../../docs/en/api/index.md) (`himodel.md` covers the four models) |
| The user guide | [`docs/en/guide.md`](../../docs/en/guide.md) |
| The history of the changes and the bit-for-bit verification record | [`docs/en/verification.md`](../../docs/en/verification.md) |
| Installation and environment setup | [`docs/en/installation.md`](../../docs/en/installation.md) |
| Unit tests (no TNG data needed) | `pytest tests/ -q` |

### What you can do next

- **Change the redshift**: changing `Z_OBS` shows how observable the same galaxy is at different distances
- **Change the model**: changing `HI_MODEL` takes all four models through the same pipeline
- **Several galaxies in one field**: upstream MARTINI's `CombinedSource` was not lifted in; stacking multiple sources has to be assembled on top of `Martini` yourself
- **A real telescope response**: replace `GaussianBeam`, or pass the WCS of an existing observation to `DataCube` (`DataCube.from_wcs`)

In [ ]:
print('=' * 66)
print('  astrovodka tutorial — complete')
print('=' * 66)
print(f'  Subhalo   {subID}   (snapshot {snapNum})')
print(f'  HI model  {HI_MODEL}')
print(f'  Observed redshift z = {Z_OBS}   D_L = {DL:.1f}   D_A = {DA:.1f}')
print(f'  M_HI      {source.m_HI_total:.4e}')
print(f'  M_H2      {source.m_H2_g.sum():.4e}')
print()
print('  Products:')
for f in ['model_diagnostics.png', 'data_products.hdf5', 'DM_HI_projections.png',
          'UV_and_RSD.png', '10arcsec_nonoise.fits', '10arcsec.fits',
          '3arcmin.fits', '10arcsec_viz.png', '3arcmin_viz.png',
          'comparison.png']:
    print(f'    sub{subID}_z{Z_OBS}_{f}')
print(f'\n  all in {OUTDIR}')